# Training — Multi-Head Expert Model (v3)

Train the 4-expert Multi-Head model with medical backbones.

**Backbones :**
1. EfficientNetV2-S — mammoscreen (entraîné RSNA breast cancer, AUC 0.945)
2. DenseNet121 — TorchXRayVision RSNA X-ray
3. ResNet50 — RadImageNet (1.35M medical images)
4. ConvNeXt-Small — ImageNet-21k (RSNA Kaggle winner)

**3 phases d'entraînement :**
- Phase 1 : backbones gelés (6.3M params)
- Phase 2 : derniers 2 blocs dégelés
- Phase 3 : fine-tuning complet (106M params)

Résultats sauvegardés dans `results/metrics/multihead_v3_*.json`

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.metrics import f1_score, recall_score, precision_score

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = os.environ.get('DATA_DIR', '/kaggle/input/rsna-breast-cancer-detection')
CODE_DIR = os.environ.get('CODE_DIR', '/kaggle/input/rsna-breast-cancer-code')
OUT_DIR  = os.environ.get('OUT_DIR',  '/kaggle/working/results')
# RadImageNet weights (optionnel — fallback ImageNet si absent)
RADIMGNET_PATH = os.environ.get('RADIMGNET_PATH', None)

sys.path.insert(0, CODE_DIR)
os.makedirs(f'{OUT_DIR}/metrics', exist_ok=True)
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUT_DIR}/checkpoints', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Load & split (patient-wise) ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split

df = pd.read_csv(f'{DATA_DIR}/train.csv')
df['density'] = df['density'].fillna('B')
print(f'{len(df)} images — {df.cancer.mean()*100:.2f}% cancer')

patients = df.groupby('patient_id')['cancer'].max().reset_index()
train_pats, tmp_pats = train_test_split(patients, test_size=0.3, stratify=patients['cancer'], random_state=42)
val_pats,  test_pats = train_test_split(tmp_pats,  test_size=0.5, stratify=tmp_pats['cancer'],  random_state=42)

df_train = df[df.patient_id.isin(train_pats.patient_id)].reset_index(drop=True)
df_val   = df[df.patient_id.isin(val_pats.patient_id)].reset_index(drop=True)
df_test  = df[df.patient_id.isin(test_pats.patient_id)].reset_index(drop=True)

print(f'Train : {len(df_train)} ({df_train.cancer.mean()*100:.1f}% cancer)')
print(f'Val   : {len(df_val)} ({df_val.cancer.mean()*100:.1f}% cancer)')
print(f'Test  : {len(df_test)} ({df_test.cancer.mean()*100:.1f}% cancer)')

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
from torch.utils.data import DataLoader
from models.dataset import MammographyDataset, BalancedPatientSampler

IMAGES_DIR  = f'{DATA_DIR}/train_images'
TARGET_SIZE = (512, 512)
BATCH_SIZE  = 8   # 106M params — batch 8 avec mixed precision sur T4

ds_train = MammographyDataset(df_train, IMAGES_DIR, target_size=TARGET_SIZE, mode='train', preprocess_mode='full')
ds_val   = MammographyDataset(df_val,   IMAGES_DIR, target_size=TARGET_SIZE, mode='val',   preprocess_mode='full')
ds_test  = MammographyDataset(df_test,  IMAGES_DIR, target_size=TARGET_SIZE, mode='test',  preprocess_mode='full')

sampler = BalancedPatientSampler(df_train['patient_id'].values, df_train['cancer'].values, pos_weight=13.7)

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=4, pin_memory=True)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=4, pin_memory=True)

print(f'Train batches : {len(train_loader)}')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
from models.multi_head_expert import MultiHeadMammoModel

model = MultiHeadMammoModel(
    embed_dim=512,
    radImageNet_densenet=None,          # optionnel
    radImageNet_resnet=RADIMGNET_PATH   # None = fallback ImageNet
).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_all   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable (all) : {trainable_all:,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — Backbones gelés (fusion + tête seulement)
# ══════════════════════════════════════════════════════════════════════════════
from models.trainer import Trainer

model.freeze_backbones()
frozen_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1 — {frozen_params:,} params entraînables (backbones gelés)')

trainer_p1 = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    device=DEVICE, lr=1e-3, n_epochs=10, patience=4,
    checkpoint_dir=f'{OUT_DIR}/checkpoints',
    pos_weight=13.7, use_amp=(DEVICE=='cuda')
)
history_p1 = trainer_p1.train()
print(f'Phase 1 terminée — best val AUROC : {max(history_p1["val_auroc"]):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — Dégeler les 2 derniers blocs
# ══════════════════════════════════════════════════════════════════════════════
model.unfreeze_backbones(last_n_blocks=2)
unfrozen_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 — {unfrozen_params:,} params entraînables')

trainer_p2 = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    device=DEVICE, lr=1e-4, n_epochs=10, patience=4,
    checkpoint_dir=f'{OUT_DIR}/checkpoints',
    pos_weight=13.7, use_amp=(DEVICE=='cuda')
)
history_p2 = trainer_p2.train()
print(f'Phase 2 terminée — best val AUROC : {max(history_p2["val_auroc"]):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3 — Fine-tuning complet
# ══════════════════════════════════════════════════════════════════════════════
model.unfreeze_all()
print(f'Phase 3 — {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params entraînables')

trainer_p3 = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    device=DEVICE, lr=5e-5, n_epochs=15, patience=5,
    checkpoint_dir=f'{OUT_DIR}/checkpoints',
    pos_weight=13.7, use_amp=(DEVICE=='cuda')
)
history_p3 = trainer_p3.train()
print(f'Phase 3 terminée — best val AUROC : {max(history_p3["val_auroc"]):.4f}')

In [ ]:
# ── Courbes d'entraînement (3 phases) ─────────────────────────────────────────
def concat_history(h1, h2, h3, key):
    return h1.get(key,[]) + h2.get(key,[]) + h3.get(key,[])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Multi-Head Expert v3 — Training curves (3 phases)', fontsize=13)

# Séparateurs de phases
p1_end = len(history_p1.get('val_auroc', []))
p2_end = p1_end + len(history_p2.get('val_auroc', []))

auroc_curve = concat_history(history_p1, history_p2, history_p3, 'val_auroc')
loss_train  = concat_history(history_p1, history_p2, history_p3, 'train_loss')
loss_val    = concat_history(history_p1, history_p2, history_p3, 'val_loss')

for ax in axes:
    ax.axvline(p1_end, color='orange', linestyle='--', alpha=0.6, label='phase 2')
    ax.axvline(p2_end, color='green',  linestyle='--', alpha=0.6, label='phase 3')

axes[0].plot(loss_train, label='train'); axes[0].plot(loss_val, label='val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(auroc_curve)
axes[1].axhline(0.66, color='red',  linestyle=':', label='v1 (0.66)')
axes[1].axhline(0.84, color='blue', linestyle=':', label='target (0.84)')
axes[1].set_title('Val ROC-AUC'); axes[1].legend()

f1_curve = concat_history(history_p1, history_p2, history_p3, 'val_f1')
axes[2].plot(f1_curve); axes[2].set_title('Val F1')

plt.tight_layout()
fig_path = f'{OUT_DIR}/figures/multihead_v3_training.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Évaluation test (patient-level) ──────────────────────────────────────────
model.eval()
all_probs, all_labels, all_pids = [], [], []

with torch.no_grad():
    for imgs, labels, pids in test_loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs).squeeze(1)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
        all_pids.extend(pids.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
all_pids   = np.array(all_pids)

pat_probs, pat_ids = MammographyDataset.patient_level_aggregate(all_probs, all_pids, 'max')
pat_labels = np.array([all_labels[all_pids == pid].max() for pid in pat_ids])

auroc = roc_auc_score(pat_labels, pat_probs)
prauc = average_precision_score(pat_labels, pat_probs)
fpr, tpr, thresholds = roc_curve(pat_labels, pat_probs)
best_thr  = thresholds[np.argmax(tpr - fpr)]
preds     = (pat_probs >= best_thr).astype(int)
f1        = f1_score(pat_labels, preds, zero_division=0)
recall    = recall_score(pat_labels, preds, zero_division=0)
precision = precision_score(pat_labels, preds, zero_division=0)

print(f'\n=== Multi-Head Expert v3 — Test set (patient-level) ===')
print(f'ROC-AUC   : {auroc:.4f}  (ref v1: 0.66 | target: 0.84)')
print(f'PR-AUC    : {prauc:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Threshold : {best_thr:.4f}')

In [ ]:
# ── Courbe ROC + comparaison v1 ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, linewidth=2, label=f'Multi-Head v3 (AUC={auroc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4)
# Référence v1 (point approximatif AUROC 0.66)
ax.axhline(0, color='none')
ax.scatter([0.3],[0.66], marker='*', s=200, color='red', zorder=5, label='Multi-Head v1 (AUC≈0.66)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Multi-Head Expert v3 (patient-level)')
ax.legend()
plt.tight_layout()
roc_path = f'{OUT_DIR}/figures/multihead_v3_roc.png'
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sauvegarde métriques JSON ─────────────────────────────────────────────────
metrics = {
    'model': 'MultiHeadExpert_v3',
    'backbones': [
        'mammoscreen (EfficientNetV2-S, RSNA breast)',
        'TorchXRayVision DenseNet121-RSNA',
        'RadImageNet ResNet50',
        'ConvNeXt-Small ImageNet-21k'
    ],
    'total_params': total_params,
    'training_phases': {
        'phase1_epochs': len(history_p1.get('val_auroc', [])),
        'phase2_epochs': len(history_p2.get('val_auroc', [])),
        'phase3_epochs': len(history_p3.get('val_auroc', []))
    },
    'best_val_auroc': round(float(max(auroc_curve)), 4),
    'test_patient_level': {
        'roc_auc':   round(float(auroc), 4),
        'pr_auc':    round(float(prauc), 4),
        'f1':        round(float(f1), 4),
        'recall':    round(float(recall), 4),
        'precision': round(float(precision), 4),
        'threshold': round(float(best_thr), 4)
    },
    'reference_v1': {'roc_auc': 0.66, 'f1': 0.22, 'recall': 0.23},
    'preprocess_mode': 'full',
    'target_size': list(TARGET_SIZE)
}

json_path = f'{OUT_DIR}/metrics/multihead_v3.json'
with open(json_path, 'w') as f:
    json.dump(metrics, f, indent=2)

delta = metrics['test_patient_level']['roc_auc'] - 0.66
print(f'Métriques sauvegardées : {json_path}')
print(f'\nDelta vs v1 : {delta:+.4f} AUROC')
print(json.dumps(metrics['test_patient_level'], indent=2))